#MOdels

In [ ]:
from pyspark.ml.classification import (
    LogisticRegression, DecisionTreeClassifier,
    RandomForestClassifier, GBTClassifier, OneVsRest
)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

acc_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)

In [ ]:
print("3-class target: PROPERTY / VIOLENT / DRUG")
print("Params: regParam=0.001, elasticNetParam=0.0, maxIter=100")

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    family="multinomial",
    regParam=0.001,      # lower regularisation for 3-class problem
    elasticNetParam=0.0,
    maxIter=100          # more iterations for better convergence
)

start_lr = time.perf_counter()
lr_model = lr.fit(train_df)
lr_time  = time.perf_counter() - start_lr

lr_pred = lr_model.transform(test_df)
lr_acc  = float(acc_eval.evaluate(lr_pred))
lr_f1   = float(f1_eval.evaluate(lr_pred))

print("Logistic Regression Accuracy:", round(lr_acc, 4))
print("Logistic Regression F1 Score:", round(lr_f1, 4))
print("Train time (seconds):", round(lr_time, 2))

lr_model.write().overwrite().save(MODEL_DIR + "/lr_model")
lr_pred.write.mode("overwrite").parquet(PRED_DIR + "/lr_predictions")

3-class target: PROPERTY / VIOLENT / DRUG
Params: regParam=0.001, elasticNetParam=0.0, maxIter=100
Logistic Regression Accuracy: 0.5992
Logistic Regression F1 Score: 0.4527
Train time (seconds): 57.65


In [ ]:
print("3-class target: PROPERTY / VIOLENT / DRUG")
print("Params: maxDepth=15, maxBins=128, minInstancesPerNode=10")

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    maxDepth=15,             # deeper tree for 22 features
    maxBins=128,             # finer splits on continuous lat/lon/geo_cell
    minInstancesPerNode=10,  # allow finer leaf nodes with 6M rows
    seed=42
)

start_dt = time.perf_counter()
dt_model = dt.fit(train_df)
dt_time  = time.perf_counter() - start_dt

dt_pred = dt_model.transform(test_df)
dt_acc  = float(acc_eval.evaluate(dt_pred))
dt_f1   = float(f1_eval.evaluate(dt_pred))

print("Decision Tree Accuracy:", round(dt_acc, 4))
print("Decision Tree F1 Score:", round(dt_f1, 4))
print("Train time (seconds):", round(dt_time, 2))
print("Tree depth:", dt_model.depth)

dt_importances = dt_model.featureImportances.toArray()
fi_dt = pd.DataFrame({
    "feature":    [c + "_imp" for c in num_cols],
    "importance": dt_importances
}).sort_values("importance", ascending=False)
print("\nDecision Tree Feature Importances:")
print(fi_dt.to_string(index=False))
fi_dt.to_csv(OUT_DIR + "/dash2_dt_feature_importance.csv", index=False)

dt_model.write().overwrite().save(MODEL_DIR + "/dt_model")
dt_pred.write.mode("overwrite").parquet(PRED_DIR + "/dt_predictions")

3-class target: PROPERTY / VIOLENT / DRUG
Params: maxDepth=15, maxBins=128, minInstancesPerNode=10
Decision Tree Accuracy: 0.5984
Decision Tree F1 Score: 0.5316
Train time (seconds): 62.87
Tree depth: 15

Decision Tree Feature Importances:
              feature  importance
        Longitude_imp    0.153548
         geo_cell_imp    0.150193
         Latitude_imp    0.141953
             Ward_imp    0.115399
         year_num_imp    0.091578
       hour_x_lon_imp    0.063210
             hour_imp    0.059320
       hour_x_lat_imp    0.036435
             Beat_imp    0.035131
            month_imp    0.029277
         lon_grid_imp    0.021875
      day_of_week_imp    0.020575
  district_x_hour_imp    0.016036
   Community Area_imp    0.014767
         lat_grid_imp    0.014243
       is_weekend_imp    0.013758
           season_imp    0.008652
         District_imp    0.003854
     ward_x_night_imp    0.003609
is_business_hours_imp    0.003416
         is_night_imp    0.002398
       is_ev

In [ ]:
print("Role: Ensemble of deeper trees. Reduces variance of single DT.")
print("Params: numTrees=150, maxDepth=15, maxBins=128, featureSubsetStrategy=sqrt")

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=150,                  # 150 trees for 3-class stability
    maxDepth=15,                   # matches DT depth
    maxBins=128,                   # finer geo splits
    minInstancesPerNode=5,
    featureSubsetStrategy="sqrt",  # sqrt(22) ~= 5 features per split
    seed=42
)

start_rf = time.perf_counter()
rf_model = rf.fit(train_df)
rf_time  = time.perf_counter() - start_rf

rf_pred = rf_model.transform(test_df)
rf_acc  = float(acc_eval.evaluate(rf_pred))
rf_f1   = float(f1_eval.evaluate(rf_pred))

print("Random Forest Accuracy:", round(rf_acc, 4))
print("Random Forest F1 Score:", round(rf_f1, 4))
print("Train time (seconds):", round(rf_time, 2))

rf_importances = rf_model.featureImportances.toArray()
fi_rf = pd.DataFrame({
    "feature":    [c + "_imp" for c in num_cols],
    "importance": rf_importances
}).sort_values("importance", ascending=False)
print("\nRandom Forest Feature Importances:")
print(fi_rf.to_string(index=False))
fi_rf.to_csv(OUT_DIR + "/dash2_rf_feature_importance.csv", index=False)

rf_model.write().overwrite().save(MODEL_DIR + "/rf_model")
rf_pred.write.mode("overwrite").parquet(PRED_DIR + "/rf_predictions")

Role: Ensemble of deeper trees. Reduces variance of single DT.
Params: numTrees=150, maxDepth=15, maxBins=128, featureSubsetStrategy=sqrt
Random Forest Accuracy: 0.6108
Random Forest F1 Score: 0.5182
Train time (seconds): 3619.79

Random Forest Feature Importances:
              feature  importance
        Longitude_imp    0.099394
         Latitude_imp    0.089264
         geo_cell_imp    0.085125
         year_num_imp    0.083552
             Beat_imp    0.067562
         lat_grid_imp    0.063741
         lon_grid_imp    0.060761
             Ward_imp    0.060306
   Community Area_imp    0.052727
       hour_x_lon_imp    0.051691
       hour_x_lat_imp    0.050971
         District_imp    0.044161
             hour_imp    0.041565
  district_x_hour_imp    0.036354
            month_imp    0.035401
      day_of_week_imp    0.028671
           season_imp    0.016979
       is_weekend_imp    0.012936
     ward_x_night_imp    0.009268
         is_night_imp    0.004670
is_business_hours_im

In [ ]:
print("Target: 1 = VIOLENT crime, 0 = PROPERTY or DRUG")
print("Params: maxIter=30, maxDepth=8, stepSize=0.05")

# Create binary label: 1 = PROPERTY, 0 = everything else
train_gbt = train_df.withColumn(
    "label_binary",
    F.when(F.col("crime_category") == "VIOLENT", 1.0).otherwise(0.0)
)
test_gbt = test_df.withColumn(
    "label_binary",
    F.when(F.col("crime_category") == "VIOLENT", 1.0).otherwise(0.0)
)

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label_binary",
    maxIter=30,       # more boosting rounds = better discrimination
    maxDepth=8,
    stepSize=0.05,    # lower learning rate = less overfitting
    seed=42
)

start_gbt = time.perf_counter()
gbt_model = gbt.fit(train_gbt)
gbt_time  = time.perf_counter() - start_gbt

gbt_pred_raw = gbt_model.transform(test_gbt)

# Evaluate GBT with binary metrics
from pyspark.ml.evaluation import BinaryClassificationEvaluator
gbt_auc_eval = BinaryClassificationEvaluator(
    labelCol="label_binary",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
gbt_auc = float(gbt_auc_eval.evaluate(gbt_pred_raw))

# Also compute accuracy and F1 on binary prediction
gbt_acc_eval = MulticlassClassificationEvaluator(
    labelCol="label_binary", predictionCol="prediction", metricName="accuracy"
)
gbt_f1_eval = MulticlassClassificationEvaluator(
    labelCol="label_binary", predictionCol="prediction", metricName="f1"
)
gbt_acc = float(gbt_acc_eval.evaluate(gbt_pred_raw))
gbt_f1  = float(gbt_f1_eval.evaluate(gbt_pred_raw))

print("GBT Binary AUC-ROC:   ", round(gbt_auc, 4))
print("GBT Binary Accuracy:  ", round(gbt_acc, 4))
print("GBT Binary F1 Score:  ", round(gbt_f1, 4))
print("Train time (seconds): ", round(gbt_time, 2))
print("Note: AUC-ROC is the primary metric for binary GBT (VIOLENT vs Other).")

Target: 1 = VIOLENT crime, 0 = PROPERTY or DRUG
Params: maxIter=30, maxDepth=8, stepSize=0.05
GBT Binary AUC-ROC:    0.6275
GBT Binary Accuracy:   0.6931
GBT Binary F1 Score:   0.5902
Train time (seconds):  223.11
Note: AUC-ROC is the primary metric for binary GBT (VIOLENT vs Other).


In [ ]:
# GBT feature importance
gbt_importances = gbt_model.featureImportances.toArray()
fi_gbt = pd.DataFrame({
    "feature":    [c + "_imp" for c in num_cols],
    "importance": gbt_importances
}).sort_values("importance", ascending=False)
print("\nGBT Feature Importances:")
print(fi_gbt.to_string(index=False))
fi_gbt.to_csv(OUT_DIR + "/dash2_gbt_feature_importance.csv", index=False)

# Rename prediction column so it doesn't clash with multi-class predictions
gbt_pred = gbt_pred_raw.withColumnRenamed("prediction", "gbt_prediction")

gbt_model.write().overwrite().save(MODEL_DIR + "/gbt_model")
gbt_pred_raw.write.mode("overwrite").parquet(PRED_DIR + "/gbt_predictions")


GBT Feature Importances:
              feature  importance
         geo_cell_imp    0.120818
         year_num_imp    0.115525
        Longitude_imp    0.087544
         lon_grid_imp    0.083461
         Latitude_imp    0.071184
            month_imp    0.059664
             hour_imp    0.057566
         lat_grid_imp    0.051737
       hour_x_lon_imp    0.050051
             Beat_imp    0.048494
             Ward_imp    0.038076
      day_of_week_imp    0.037255
       hour_x_lat_imp    0.034666
  district_x_hour_imp    0.034140
   Community Area_imp    0.031965
           season_imp    0.015448
is_business_hours_imp    0.013896
       is_weekend_imp    0.012061
     ward_x_night_imp    0.011822
         is_night_imp    0.011427
         District_imp    0.010755
       is_evening_imp    0.002444
